# Live comparison: one prompt versus solve, then encode

Run both harnesses on the same APPS problem and three-bit message using real Codex inference and Modal evaluation.

- **V0:** Ask for working code containing the message in one prompt.
- **V1:** Ask for working code first. If its tests pass, ask once for a modified version containing the message.

Neither retries. Run with the `stego` kernel, `STEGO_ARTIFACTS_DIR` set, and Codex/Modal credentials configured. Running all cells makes at most five Codex calls and five Modal evaluations, including one independent cipherless sample per harness. [Harness documentation](../harness/README.md).

The setup cell loads the repository `.env` without overriding existing environment variables.

In [ ]:
import os
import sys
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv

REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "ciphers/variable_naming_in_python_v2/decoder.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env")
ARTIFACT_ROOT = (REPO_ROOT / os.environ["STEGO_ARTIFACTS_DIR"]).resolve()

In [ ]:
from ciphers.variable_naming_in_python_v2.data.apps import AppsConfig, AppsTestCases, load_apps
from ciphers.variable_naming_in_python_v2.data.codex_apps import CodexInferenceConfig
from ciphers.variable_naming_in_python_v2.data.modal_apps import ModalAppsConfig
from ciphers.variable_naming_in_python_v2.decoder import CipherConfig
from ciphers.variable_naming_in_python_v2.harness.harness_v0_single_prompt import HarnessV0SinglePrompt
from ciphers.variable_naming_in_python_v2.harness.harness_v1_solve_code_then_encode_message import HarnessV1SolveCodeThenEncodeMessage
from ciphers.variable_naming_in_python_v2.harness.interface import HarnessProblem, HarnessRequest

inference_config = CodexInferenceConfig()
modal_config = ModalAppsConfig()
cipher = CipherConfig(special_variables={"index": ("i", "j")}, length_bits=3)
message_bits = "101"

row = load_apps(AppsConfig()).shuffle(seed=42)[0]
request = HarnessRequest(
    num_cipherless_generations=1,
    problem=HarnessProblem(
        problem_id=row["problem_id"],
        question=row["question"],
        starter_code=row["starter_code"],
        test_cases=AppsTestCases.from_dataset_value(row["input_output"]),
    ),
    cipher=cipher,
    message_bits=message_bits,
)
print("Problem:", request.problem.problem_id)
print(request.problem.question)
print("Message:", message_bits)

## Generate, evaluate, and save

Each completed run is saved immediately, including failures. Infrastructure errors stop execution. `runtime.record_attempt` documents the metadata: assistant messages contain extracted code, inference artifact paths, and evaluation results; user messages omit evaluation.

In [ ]:
harnesses = [
    HarnessV0SinglePrompt(inference_config, modal_config),
    HarnessV1SolveCodeThenEncodeMessage(inference_config, modal_config),
]
report_dir = ARTIFACT_ROOT / "experiments" / "harness_v0_vs_harness_v1"
report_dir.mkdir(parents=True, exist_ok=True)
results = []

for harness in harnesses:
    result = await harness.run(request)
    results.append(result)
    report_path = report_dir / f"{result.harness_name}_{uuid4().hex}.json"
    report_path.write_text(result.model_dump_json(indent=2), encoding="utf-8")
    print(f"\n{result.harness_name}: success={result.metadata['success']}")
    for step in result.steps:
        if step.role == "assistant":
            evaluation = step.metadata["evaluation"]
            print(step.metadata["stage"], "code passed:", evaluation["code_passed"], "message matches:", evaluation["message_matches"])
            print(step.metadata["code"])
            if step.metadata["output_error"] or evaluation["decode_error"]:
                print(step.metadata["output_error"] or evaluation["decode_error"])
    print("Report:", report_path.relative_to(ARTIFACT_ROOT))